# SmokeWatch — ré-entraînement d'un modèle

Carnet prévu pour **Google Colab** ou **Kaggle**, GPU gratuit suffisant.

## Le principe, en une phrase

On ne repart **jamais de zéro** : on part des poids actuels du modèle et on lui
montre ce qu'il rate et ce qu'il croit voir à tort. C'est du rattrapage, pas de
la rééducation — d'où quelques dizaines de minutes plutôt que plusieurs jours.

## Trois règles à ne pas enfreindre

1. **Rester en 640 pixels.** Les modèles OpenVINO du serveur ont cette taille
   figée. Entraîner en 416 ou 832 obligerait à tout réexporter et ferait planter
   l'inférence — ce piège a déjà coûté une soirée sur ce projet.
2. **Sauvegarder sur Drive.** Colab coupe sans prévenir. Sans point de reprise,
   deux heures d'entraînement disparaissent.
3. **Mesurer avant et après.** Sans comparaison, on ne sait pas si l'on a
   progressé ou reculé — et un modèle qui recule ne se voit nulle part, parce
   qu'un manque ne s'affiche pas.

---


## Étape 1 — Vérifier le GPU

Dans Colab : *Exécution → Modifier le type d'exécution → GPU T4*.
Sans GPU l'entraînement reste possible, mais comptez dix fois plus de temps.


In [ ]:
!nvidia-smi

import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Carte :', torch.cuda.get_device_name(0))


## Étape 2 — Installer Ultralytics


In [ ]:
!pip install -q ultralytics roboflow

from ultralytics import YOLO
import ultralytics
ultralytics.checks()


## Étape 3 — Monter Google Drive

Sert à deux choses : récupérer les poids actuels du modèle, et **survivre à une
coupure de Colab**. Déposez au préalable votre `.pt` dans
`MonDrive/SmokeWatch/modeles/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/SmokeWatch')
(DRIVE / 'modeles').mkdir(parents=True, exist_ok=True)
(DRIVE / 'runs').mkdir(parents=True, exist_ok=True)

print('Modèles présents sur le Drive :')
for f in sorted((DRIVE / 'modeles').glob('*.pt')):
    print(' -', f.name, f'({f.stat().st_size / 1e6:.1f} Mo)')


## Étape 4 — Choisir le modèle à corriger

Modifiez la variable ci-dessous, **une seule** à la fois. L'ordre recommandé :

| Priorité | Modèle | Défaut constaté | Pourquoi cet ordre |
|---|---|---|---|
| 1 | `epi` | rappel ~54 % sur NO-Hardhat | un ouvrier sans casque sur deux passe inaperçu, et **un manque ne s'affiche pas** |
| 2 | `load_control` | affirme `empty` à 0,89 hors contexte | corrigeable uniquement par des images négatives |
| 3 | `gloves_glasses` | chute peu fiable | aujourd'hui masquée par un seuil, pas corrigée |
| 4 | `plate` | modèle inexistant | améliorerait nettement la lecture des plaques |


In [ ]:
# ── À MODIFIER ──────────────────────────────────────────────
MODELE = 'epi'          # epi | load_control | gloves_glasses | plate | conveyor
EPOCHS = 40             # 40 suffit pour un rattrapage ; 100+ pour un modèle neuf
IMGSZ  = 640            # NE PAS CHANGER : taille figée côté serveur
BATCH  = 16             # réduire à 8 si « CUDA out of memory »
# ────────────────────────────────────────────────────────────

POIDS_ACTUELS = DRIVE / 'modeles' / f'smokewatch_{MODELE}_best.pt'
NEUF = not POIDS_ACTUELS.exists()

if NEUF:
    print(f"Aucun poids existant pour '{MODELE}' : entraînement depuis yolov8n.")
    print('Normal pour plate et conveyor, anormal pour les autres.')
    DEPART = 'yolov8n.pt'
else:
    print(f'Rattrapage à partir de {POIDS_ACTUELS.name}')
    DEPART = str(POIDS_ACTUELS)


## Étape 5 — Récupérer le jeu de données

Créez un compte gratuit sur [Roboflow](https://roboflow.com), puis récupérez
votre clé dans *Settings → API Keys*.

| Modèle | Jeu | Volume | Classes |
|---|---|---|---|
| `epi` | [HardHat & SafetyVest](https://universe.roboflow.com/ppe-kit-detection/hardhat-safetyvest) | 22 068 | casque, gilet, absences |
| `fall` | [Fall Detection](https://universe.roboflow.com/roboflow-universe-projects/fall-detection-ca3o8) | 4 497 | `standing`, `bending`, `falling`, `fallen` |
| `plate` | [Plaques marocaines](https://universe.roboflow.com/naima-el-menani/moroccan-license-plate-detection) | 2 588 | plaque |
| `load_control` | [overloaded detection](https://universe.roboflow.com/overloaded-vehicle/overloaded-detection) | 150 | `overloaded`, `notoverloaded` |

### Pour la chute : ce jeu répond exactement au défaut actuel

Il distingue **`bending`** (penché) de **`fallen`** (à terre). C'est
précisément ce que le modèle actuel ne sait pas faire : il alerte sur
quelqu'un d'assis ou penché. Un modèle qui apprend les quatre postures
apprend aussi à **ne pas** alerter sur les trois premières.

### Pour le contrôle de sortie : fusionner dans Roboflow

Aucun jeu ne couvre le cas à lui seul. Trois s'en approchent :

- [overloaded detection](https://universe.roboflow.com/overloaded-vehicle/overloaded-detection) — 150 images, `overloaded` / `notoverloaded`
- [Overloaded Truck](https://universe.roboflow.com/mochammad-eka/overloaded-truck) — 602 images
- [tarp detection](https://universe.roboflow.com/stuff-bz6cc/tarp-detection) — 477 images de bâches

**Ne fusionnez pas ces jeux par script.** Leurs classes portent des noms et
des indices différents ; une fusion à la main produirait un modèle qui
confond les catégories, et l'erreur ne se verrait qu'à la fin.

Créez plutôt un projet dans Roboflow, importez-y les trois jeux **et vos
images de portail**, puis renommez les classes vers votre nomenclature :
`bache_absente`, `bache_partielle`, `bache_dechiree`, `surcharge`,
`conforme`. Roboflow gère la correspondance des classes, c'est son métier.
Renseignez ensuite `MON_PROJET` dans la cellule suivante.

Les images publiques apprennent au modèle à quoi ressemblent une bâche et
une surcharge ; **vos images lui apprennent votre portail**. Les deux sont
nécessaires, et les vôtres pèsent plus lourd.


In [ ]:
from roboflow import Roboflow

CLE_ROBOFLOW = ''   # ← votre clé API

# Jeux publics vérifiés. Pour load_control, aucun jeu ne couvre le cas à lui
# seul : voir la note ci-dessus, la fusion se fait dans Roboflow.
JEUX = {
    'epi':            ('ppe-kit-detection', 'hardhat-safetyvest'),
    'fall':           ('roboflow-universe-projects', 'fall-detection-ca3o8'),
    'gloves_glasses': ('roboflow-universe-projects', 'fall-detection-ca3o8'),
    'plate':          ('naima-el-menani', 'moroccan-license-plate-detection'),
    'load_control':   ('mochammad-eka', 'overloaded-truck'),
}

# Si vous avez constitué votre propre projet Roboflow (recommandé pour
# load_control), renseignez-le ici et il primera sur le jeu public.
MON_PROJET = None   # ex. ('mon-espace', 'controle-sortie-camions')

source = MON_PROJET or JEUX.get(MODELE)
if source and CLE_ROBOFLOW:
    espace, projet = source
    rf = Roboflow(api_key=CLE_ROBOFLOW)
    p = rf.workspace(espace).project(projet)
    dataset = p.version(max(v.version for v in p.versions())).download('yolov8')
    CHEMIN_DATASET = dataset.location
    print('Jeu de données téléchargé dans', CHEMIN_DATASET)
    import yaml
    conf = yaml.safe_load(open(f'{CHEMIN_DATASET}/data.yaml'))
    print('Classes :', conf.get('names'))
else:
    CHEMIN_DATASET = None
    print('Pas de jeu public pour ce modèle, ou clé manquante.')
    print("Passez à l'étape 6 : vos propres images.")


## Étape 6 — Ajouter vos images de production

**C'est l'étape qui change tout, et elle n'a pas d'équivalent public.**

Sur le serveur SmokeWatch :

```powershell
.\venv\Scripts\python.exe scripts\export_dataset.py --model epi --days 90
```

Ce script produit un dossier contenant :

- les alertes **marquées fausses** par les opérateurs, sans annotation — ce sont
  des **images de fond**, et c'est ainsi qu'on apprend à un modèle à répondre
  « rien ici ». C'est exactement ce qui manque à `load_control`, qui n'a jamais
  vu de scène sans chargement ;
- les alertes justes, **pré-annotées** à partir des positions enregistrées.

Compressez ce dossier, déposez-le dans `MonDrive/SmokeWatch/datasets/`, puis
exécutez la cellule suivante. Vos images du site valent plus que mille images
génériques : elles montrent vos caméras, votre lumière, vos ouvriers.


In [ ]:
import shutil, zipfile, yaml

ARCHIVE_PERSO = DRIVE / 'datasets' / f'{MODELE}_production.zip'

if ARCHIVE_PERSO.exists():
    dest = Path('/content/perso')
    shutil.rmtree(dest, ignore_errors=True)
    with zipfile.ZipFile(ARCHIVE_PERSO) as z:
        z.extractall(dest)
    print('Images de production extraites.')

    if CHEMIN_DATASET:
        # Fusion : les images de production rejoignent le jeu public.
        images_src = list((dest).rglob('images/train/*'))
        for img in images_src:
            shutil.copy2(img, Path(CHEMIN_DATASET) / 'train' / 'images' / img.name)
            label = img.parent.parent.parent / 'labels' / 'train' / (img.stem + '.txt')
            if label.exists():
                shutil.copy2(label, Path(CHEMIN_DATASET) / 'train' / 'labels' / label.name)
        print(f'{len(images_src)} image(s) de production ajoutées au jeu public.')
    else:
        CHEMIN_DATASET = str(next(dest.glob('**/data.yaml')).parent)
        print('Entraînement sur vos seules images :', CHEMIN_DATASET)
else:
    print('Aucune archive de production trouvée — entraînement sur le jeu public seul.')
    print(f"Attendu : {ARCHIVE_PERSO}")


## Étape 7 — Mesurer AVANT

Le chiffre de départ. Sans lui, impossible d'affirmer quoi que ce soit à la fin
— et « ça a l'air mieux » n'est pas une phrase de soutenance.


In [ ]:
import json

AVANT = None
if not NEUF and CHEMIN_DATASET:
    modele_avant = YOLO(str(POIDS_ACTUELS))
    m = modele_avant.val(data=f'{CHEMIN_DATASET}/data.yaml', imgsz=IMGSZ, verbose=False)
    AVANT = {
        'mAP50': round(float(m.box.map50), 4),
        'mAP50-95': round(float(m.box.map), 4),
        'precision': round(float(m.box.mp), 4),
        'rappel': round(float(m.box.mr), 4),
    }
    print('AVANT :', json.dumps(AVANT, indent=2))
    print()
    print('Le RAPPEL est le chiffre à surveiller : il dit la part des cas réels')
    print('que le modèle voit. Un rappel bas = des ouvriers manqués en silence.')
else:
    print('Modèle neuf ou jeu absent : pas de mesure de départ.')


## Étape 8 — Entraîner

Comptez 20 à 60 minutes sur un T4 pour 40 époques sur quelques milliers
d'images. Le carnet reprend automatiquement si Colab a coupé en cours de route.


In [ ]:
import os

NOM_RUN = f'{MODELE}_v2'
DOSSIER_RUN = DRIVE / 'runs' / NOM_RUN
REPRISE = (DOSSIER_RUN / 'weights' / 'last.pt').exists()

modele = YOLO(str(DOSSIER_RUN / 'weights' / 'last.pt') if REPRISE else DEPART)
if REPRISE:
    print('Reprise après interruption.')

resultats = modele.train(
    data=f'{CHEMIN_DATASET}/data.yaml',
    epochs=EPOCHS,
    imgsz=IMGSZ,              # 640 — ne pas changer
    batch=BATCH,
    resume=REPRISE,
    project=str(DRIVE / 'runs'),   # écriture directe sur Drive
    name=NOM_RUN,
    exist_ok=True,
    patience=15,              # arrêt si aucun progrès pendant 15 époques
    save_period=5,            # point de reprise toutes les 5 époques
    # Augmentations utiles en surveillance : lumière et point de vue varient
    # d'une caméra à l'autre, pas l'orientation verticale.
    hsv_v=0.5,                # variations d'éclairage
    degrees=8,                # légère inclinaison de caméra
    fliplr=0.5,               # symétrie gauche/droite
    flipud=0.0,               # jamais à l'envers : personne ne marche au plafond
    mosaic=1.0,
)
print('Entraînement terminé.')


## Étape 9 — Mesurer APRÈS et comparer

Le moment de vérité. **Si le rappel a baissé, ne déployez pas** : vous auriez
rendu le système plus silencieux, pas meilleur.


In [ ]:
NOUVEAUX_POIDS = DOSSIER_RUN / 'weights' / 'best.pt'

modele_apres = YOLO(str(NOUVEAUX_POIDS))
m = modele_apres.val(data=f'{CHEMIN_DATASET}/data.yaml', imgsz=IMGSZ, verbose=False)
APRES = {
    'mAP50': round(float(m.box.map50), 4),
    'mAP50-95': round(float(m.box.map), 4),
    'precision': round(float(m.box.mp), 4),
    'rappel': round(float(m.box.mr), 4),
}

print(f"{'Mesure':<12}{'avant':>10}{'après':>10}{'écart':>10}")
print('-' * 42)
for cle in APRES:
    a = AVANT[cle] if AVANT else None
    ecart = f'{APRES[cle] - a:+.4f}' if a is not None else '—'
    print(f"{cle:<12}{(a if a is not None else '—'):>10}{APRES[cle]:>10}{ecart:>10}")

print()
if AVANT and APRES['rappel'] < AVANT['rappel']:
    print('⚠ LE RAPPEL A BAISSÉ. Le modèle rate désormais PLUS de cas réels.')
    print('  Ne déployez pas : vous rendriez le système silencieux, pas meilleur.')
elif AVANT:
    gain = (APRES['rappel'] - AVANT['rappel']) * 100
    print(f'Rappel : {AVANT["rappel"]*100:.1f} % → {APRES["rappel"]*100:.1f} % ({gain:+.1f} points)')
    print('Cette phrase-là a sa place dans un mémoire.')


## Étape 10 — Voir les résultats par classe

La moyenne cache l'essentiel. Sur `epi`, c'est `NO-Hardhat` qui compte, pas la
moyenne des onze classes.


In [ ]:
noms = modele_apres.names
print(f"{'classe':<22}{'précision':>11}{'rappel':>9}{'mAP50':>9}")
print('-' * 51)
for i, nom in noms.items():
    try:
        p, r, ap50, _ = m.box.class_result(i)
        marque = '  ←' if nom.startswith('NO-') or nom in ('crack', 'torn', 'empty') else ''
        print(f'{nom:<22}{p:>11.3f}{r:>9.3f}{ap50:>9.3f}{marque}')
    except Exception:
        pass
print()
print('Les classes marquées ← sont celles qui déclenchent une alerte.')
print('Un rappel bas sur ces lignes est un risque, pas une statistique.')


## Étape 11 — Exporter et déployer

Le nouveau `.pt` doit être **reconverti en OpenVINO** sur le serveur, sinon le
pipeline continuera d'utiliser l'ancien modèle — silencieusement.


In [ ]:
import shutil

final = DRIVE / 'modeles' / f'smokewatch_{MODELE}_best_v2.pt'
shutil.copy2(NOUVEAUX_POIDS, final)
print('Modèle enregistré sur le Drive :', final)

# Export OpenVINO possible ici, mais la version d'OpenVINO de Colab peut
# différer de celle du serveur : mieux vaut réexporter sur place.
print()
print('Sur le serveur SmokeWatch :')
print(f'  1. copier {final.name} dans models\\smokewatch_{MODELE}_best.pt')
print('     (garder l\'ancien de côté : il faut pouvoir revenir en arrière)')
print(f'  2. supprimer models\\smokewatch_{MODELE}_best_openvino_model\\')
print('  3. .\\venv\\Scripts\\python.exe scripts\\export_openvino.py')
print('  4. .\\venv\\Scripts\\python.exe scripts\\benchmark.py run')
print('  5. .\\venv\\Scripts\\python.exe scripts\\benchmark.py compare --last')
print('  6. redémarrer le pipeline')


---

## Après le déploiement

Le banc de test mesure sur **vos** vidéos, pas sur le jeu d'entraînement. Les
deux mesures se complètent : la mAP dit ce que le modèle a appris, le banc dit
ce qu'il fait sur votre site.

Si les deux progressent, le travail est fait. Si la mAP monte mais que le banc
recule, le modèle s'est spécialisé sur le jeu public au détriment de vos scènes
— ajoutez davantage d'images de production et recommencez.

## Et ensuite

Le bouton « fausse alerte » continue d'alimenter le jeu de données pendant
l'exploitation. Un second passage trois mois plus tard, avec les erreurs
réellement constatées sur site, vaut mieux qu'un premier passage parfait.
